# EXP-009: Tinh chỉnh tham số Random Forest

- Ngày: 22/11/2026 (tổng hợp notebook).
- Ngày chạy: 20/11/2026. Phần 20 cây ngày 19/11 và 100 cây ngày 22/11.
- Mã thực nghiệm: exp009.
- So với lần trước: exp005 (chọn tham số bằng inner CV).

**Thực nghiệm tương ứng:** [experiment.ipynb](../../experiments/02_tuning/EXP-009_RF/experiment.ipynb).

**Mạch thực nghiệm:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).


## 1. Mục tiêu và giả thuyết

**Mục tiêu:** Kiểm tra xem chỉnh tham số RF có giúp dự đoán trên người mới tốt hơn EXP-005 không.

**Giả thuyết:** Kỳ vọng chọn số cây, độ sâu và số biến xét tại mỗi nhánh bằng dữ liệu train có thể tăng Macro-F1. Chưa đặt mức tăng cụ thể khi chưa có kết quả.


## 2. Cấu hình

Dùng cấu hình trong `experiment.ipynb` và cấu hình đã lưu khi chạy.

So sánh với RF mặc định 100 cây của EXP-005. Giữ dữ liệu và seed 42. Không cân bằng lớp.

| Ứng viên | Số cây | Độ sâu tối đa | max_features |
| --- | ---: | --- | --- |
| C1 | 100 | 10 | sqrt |
| C2 | 200 | Không giới hạn | sqrt |
| C3 | 100 | Không giới hạn | 0.5 |
| C4 | 200 | 10 | 0.5 |

Trong mỗi outer fold, chọn cấu hình trong C1–C4 bằng 3 inner fold theo người. Các người test có thể ứng với các tham số được chọn khác nhau.

**Phần A/A+B/A+B+C:** dùng A không giới hạn độ sâu và đặt `sqrt`. Với A+B, giới hạn độ sâu 8. Với A+B+C, thêm `max_features=0.25`. Chạy mỗi cấu hình ở 20 và 100 cây. So sánh các cấu hình cố định này riêng. Không dùng chúng để thay thế bước chọn tham số C1–C4.


## 3. Cách thực hiện

Dùng dữ liệu physiological68 gồm 6.189 cửa sổ, 68 đặc trưng và 15 người. Bài toán nhị phân phân biệt không stress/stress. Bài toán ba lớp gồm baseline/stress/amusement.

Trong mỗi outer fold, giữ riêng 1 người test. Dùng 14 người train để so sánh C1–C4 bằng 3 inner fold. Chọn theo Macro-F1 rồi refit trên 14 người. Chạy 360 inner fit và 30 outer refit. Không dùng người test để chọn tham số. Phần A/A+B/A+B+C có 90 fold mô hình cho mỗi mức số cây.

Chỉ dùng train để học cách xử lý dữ liệu. Tính Accuracy, Macro-F1 và Balanced Accuracy cho từng người test. Sau đó lấy trung bình đều qua 15 người.


## 4. Kết quả

Accuracy là tỷ lệ dự đoán đúng. Macro-F1 là trung bình F1 các lớp. Balanced Accuracy là trung bình recall các lớp. 

| Bài toán | Phiên bản | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|
| binary | rf | 0.8834 | 0.8655 | 0.8782 |
| multiclass | rf | 0.7765 | 0.6540 | 0.6731 |


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

here = Path.cwd()
repo = next(p for p in (here, *here.parents) if (p / 'src/wesad/evaluation.py').is_file())
source = repo / 'outputs/exp009_rf_mai_20261120_report'
file = source / ('results.csv' if (source / 'results.csv').exists() else 'metrics_by_subject.csv')
scores = pd.read_csv(file)
if 'model' in scores:
    scores = scores.loc[scores.model.eq('rf')]
if 'variant' not in scores:
    scores['variant'] = 'rf'
keys = ['task', 'variant']
display(scores.groupby(keys)[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std']).round(4))


macro_f1         balanced_accuracy        
                       mean     std              mean     std
task       variant                                           
binary     rf        0.8655  0.1412            0.8782  0.1141
multiclass rf        0.6540  0.1388            0.6731  0.1348

### RF mặc định và tuning

![RF mặc định và tuning](../../experiments/02_tuning/EXP-009_RF/figures/default_vs_tuned.png)


### Train và test của RF tuning

![Train và test của RF tuning](../../experiments/02_tuning/EXP-009_RF/figures/tuned_train_vs_test.png)


### Cấu hình được chọn trong từng fold

![Cấu hình được chọn trong từng fold](../../experiments/02_tuning/EXP-009_RF/figures/candidate_frequency.png)


### Phần riêng — Cấu hình cố định A/A+B/A+B+C ở 20 và 100 cây

Ghi riêng kết quả các cấu hình cố định. Không gộp chúng với kết quả nested tuning C1–C4. Mỗi mức số cây có 90 fold mô hình.

**20 cây**

| task | stage | train_macro_f1 | test_macro_f1 | test_balanced_accuracy |
|---|---|---|---|---|
| binary | A | 1.0000 | 0.8678 | 0.8756 |
| binary | A+B | 0.9999 | 0.8624 | 0.8711 |
| binary | A+B+C | 0.9997 | 0.8813 | 0.8933 |
| multiclass | A | 1.0000 | 0.6278 | 0.6445 |
| multiclass | A+B | 0.9977 | 0.6414 | 0.6641 |
| multiclass | A+B+C | 0.9969 | 0.6147 | 0.6477 |

![Train và test theo cấu hình](../../experiments/02_tuning/EXP-009_RF/figures/rf_cumulative_train_test.png)

**100 cây**

| task | stage | train_macro_f1 | test_macro_f1 | test_balanced_accuracy |
|---|---|---|---|---|
| binary | A | 1.0000 | 0.8735 | 0.8865 |
| binary | A+B | 0.9999 | 0.8749 | 0.8864 |
| binary | A+B+C | 0.9999 | 0.8922 | 0.9007 |
| multiclass | A | 1.0000 | 0.6480 | 0.6700 |
| multiclass | A+B | 0.9986 | 0.6320 | 0.6561 |
| multiclass | A+B+C | 0.9974 | 0.6226 | 0.6398 |

![Train và test theo cấu hình](../../experiments/02_tuning/EXP-009_RF/figures/rf_cumulative_train_test_100.png)


## 5. So sánh với các lần trước

Dùng điểm test trong bảng. Chỉ so sánh các kết quả trong cùng bài toán.

| Bài toán | Lần chạy | Thay đổi | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|---|
| binary | exp005 | RF mặc định | 0.8900 | 0.8735 | 0.8865 |
| multiclass | exp005 | RF mặc định | 0.7717 | 0.6480 | 0.6700 |
| binary | exp009 | Chọn tham số bằng inner CV | 0.8834 | 0.8655 | 0.8782 |
| multiclass | exp009 | Chọn tham số bằng inner CV | 0.7765 | 0.6540 | 0.6731 |


## 6. Nhận xét

- **Giả thuyết:** thấy kết quả chỉ đúng một phần kỳ vọng. Nested tuning tăng F1 ba lớp khoảng 0,0060 nhưng giảm nhị phân khoảng 0,0080 so với EXP-005.
- **Quan sát bất thường:** thấy train gần 1 nhưng test thấp hơn. Chênh lệch rõ nhất ở ba lớp. Các fold chọn tham số khác nhau. Trong phần cấu hình cố định, tăng số cây nhưng không phải chỉ số nào cũng tăng.
- **Lỗi / khó khăn:** dễ lẫn nested tuning C1–C4 với A/A+B/A+B+C. Chỉ lấy tham số đã chọn bằng inner CV cho EXP-014. Tính thêm điểm train của tuning. 12.378 dự đoán test của mô hình refit khớp kết quả cũ. Chưa kiểm định ý nghĩa thống kê cho các chênh lệch.


## 7. Bước tiếp theo

1. Sang EXP-014. Giữ tham số đã chọn của từng fold và thử class_weight="balanced".
2. Xem lỗi theo lớp và theo người để hiểu các trường hợp giảm điểm.
3. Nếu mở rộng tuning, thử min_samples_leaf và các mức max_depth khác chỉ trong inner CV.

**Liên kết:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).
